In [6]:
import pandas as pd 
import numpy as np
import random
from datetime import datetime,timedelta

In [8]:
num_customers = 1000
max_invoices_per_customer = 5

today = datetime(2025, 3, 1)

sectors = [
    "Üretim ve Sanayi",
    "Tekstil ve Hazır Giyim",
    "Otomotiv ve Yan Sanayi",
    "Makine ve Ekipman Üretimi",
    "Kimya ve Plastik",
    "Ambalaj Sanayi",
    "Gıda Üretimi",
    "Mobilya ve Yapı Malzemeleri",

    "Toptan Ticaret",
    "Distribütörlük",
    "İthalat ve İhracat",
    "E-Ticaret Tedarikçileri",

    "İnşaat ve Taahhüt",
    "Müteahhitlik",
    "Altyapı Projeleri",
    "Taşeron Hizmetler",

    "Lojistik ve Taşımacılık",
    "Kara Taşımacılığı",
    "Deniz Taşımacılığı",
    "Hava Taşımacılığı",
    "Depolama ve Lojistik Hizmetleri",
    "Kargo ve Kurye",

    "Sağlık ve Medikal",
    "Özel Hastaneler",
    "Tıbbi Cihaz ve Ekipman",
    "İlaç Distribütörleri",
    "Laboratuvar Hizmetleri",

    "Hizmet Sektörü (B2B)",
    "Temizlik Hizmetleri",
    "Güvenlik Hizmetleri",
    "Catering Hizmetleri",
    "Danışmanlık",
    "Reklam ve Ajanslar",
    "IT ve Yazılım Hizmetleri",
    "İnsan Kaynakları ve Outsourcing",

    "Enerji ve Altyapı",
    "Elektrik Üretim ve Dağıtım",
    "Yenilenebilir Enerji",
    "Doğalgaz ve Altyapı Hizmetleri",

    "Turizm ve Organizasyon (B2B)",
    "Tur Operatörleri",
    "Etkinlik Organizasyonları",
    "Kurumsal Otel Hizmetleri",

    "Telekom ve Teknoloji",
    "Telekomünikasyon Altyapı",
    "Yazılım Firmaları",
    "Donanım ve IT Ekipmanları",
    "Veri Merkezi ve Bulut Hizmetleri"
]
cities = ["Trabzon", "Afyon", "Antalya" ,"Ankara", "Istanbul", "Izmir", "Bursa", "Adana", "Gaziantep", "Konya" , "Mersin" , "Sanlıurfa", "Corum" , "Hatay" , "Sivas" , "Canakkale" , "Eskişehir" , "Van" , "Aydin" , "Kocaeli" , "Tekirdag" , "Denizli" , "Malatya" , "Zonguldak" ,"Mugla" , "Erzurum" , "Edirne" , "Diyarbakır" , "Ordu", "Ankara" ,"Manisa" , "Sakarya" , "Balıkesir" ,"Kayseri ", "Samsun"] 

risk_distribution = ["Low"] * 5 + ["Medium"] * 3 + ["High"] * 2


In [9]:
#customers

customers = []
for i in range(num_customers):
    customer_id = f"C{i:04d}"
    customer_name = f"Customer_{i:04d}"
    risk = random.choice(risk_distribution)

    if risk == "Low":
        delay =np.random.randint(0,5)
    elif risk == "Medium":
        delay = np.random.randint(5,20)
    else:
            delay = np.random.randint(20,60)
    
    customers.append({
        "customer_id": customer_id,
        "sector": random.choice(sectors),
        "city": random.choice(cities),
        "risk_segment": risk,
        "avg_payment_delay": delay
    })

    customers_df =pd.DataFrame(customers)

In [ ]:
#invoices

invoices = []

invoice_id_counter = 1

for _, cust in customers_df.iterrows():
    num_invoices = random.randint(1, max_invoices_per_customer)

    for _ in range(num_invoices):
        amount = np.random.randint(5000, 50000)

        issue_date = today - timedelta(days=np.random.randint(30, 120))
        due_date = issue_date + timedelta(days=30)

        
        base_delay = cust["avg_payment_delay"]
        amount_factor = int(amount / 10000)

        actual_delay = base_delay + np.random.randint(0, amount_factor + 5)

        payment_date = due_date + timedelta(days=actual_delay)

        paid = random.random() < (
            0.9 if cust["risk_segment"] == "Low"
            else 0.6 if cust["risk_segment"] == "Medium"
            else 0.3
        )

        invoices.append({
            "invoice_id": f"INV{invoice_id_counter:05d}",
            "customer_id": cust["customer_id"],
            "amount": amount,
            "issue_date": issue_date,
            "due_date": due_date,
            "payment_date": payment_date if paid else None,
            "paid": int(paid)
        })

        invoice_id_counter += 1

invoices_df = pd.DataFrame(invoices)

In [ ]:
#collection
collections = [] 

contact_id_counter = 1

for _, inv in invoices_df.iterrows():

    if inv["paid"] == 1:
        num_contacts = np.random.randint(1, 4)
    else:
        num_contacts = np.random.randint(3, 7)

    for i in range(num_contacts):

        contact_type = "CALL" if random.random() > 0.5 else "SMS"

        contact_date = inv["due_date"] + timedelta(days=i * 3)

        
        success_prob = 0.6 if contact_type == "CALL" else 0.3

        promise = random.random() < success_prob

        collections.append({
            "contact_id": f"CNT{contact_id_counter:06d}",
            "invoice_id": inv["invoice_id"],
            "contact_type": contact_type,
            "contact_date": contact_date,
            "promise_to_pay": int(promise)
        })

        contact_id_counter += 1

collections_df = pd.DataFrame(collections)



In [ ]:
#save 

customers_df.to_csv("customers.csv", index=False)
invoices_df.to_csv("invoices.csv", index=False)
collections_df.to_csv("collections.csv", index=False)

print("Datasets created:")
print("customers.csv, invoices.csv, collections.csv")
